# Amplitude amplification: reflections, predicates, and query cost

A controlled implementation of Grover search and general amplitude amplification. We compare complete complex statevectors with an independent vector-reflection reference, then compare seeded measurements with exact probabilities. The 4-Queens example computes a reversible constraint predicate; 5-Queens retains the original pre-enumerated lookup demonstration. All execution here uses an ideal simulator.


In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'amplification.py').is_file())
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from amplification import (uniform_preparation, oracle_for_targets, diffusion,
    first_peak_iterations, analytic_probability)
from experiments import run_experiments
report = run_experiments()
print(json.dumps({'environment': report['environment'], 'scope': report['scope']}, indent=2))


## 1. Two reflections and the invariant plane

Let $A|0\rangle=|\psi\rangle=\sin\theta|g\rangle+\cos\theta|b\rangle$, where $p=\sin^2\theta$ is the initial probability of a marked outcome. For a phase oracle $O_\chi=I-2\Pi_g$ and $D_A=2|\psi\rangle\langle\psi|-I$, one iteration is $Q=D_AO_\chi$. In the ordered basis $(|g\rangle,|b\rangle)$,

$$Q=\begin{pmatrix}\cos2\theta&\sin2\theta\\-\sin2\theta&\cos2\theta\end{pmatrix},\qquad P_k=\sin^2((2k+1)\theta).$$

The optimum used here is the better nonnegative integer adjacent to the first continuous peak $\pi/(4\theta)-1/2$; ties choose fewer oracle calls. Empty/full marked sets use zero iterations. More iterations can undo amplification. Choosing this peak assumes known $p$; unknown-success search requires an additional strategy.

Qiskit strings are written $q_{n-1}\ldots q_0$. Mask zero controls with X, apply a controlled Z, then uncompute the masks. The diffuser applies $A^\dagger$, flips the zero-state phase, then applies $A$ and a global phase $\pi$, giving the exact sign convention above.


In [ ]:
# The one-qubit case must yield X, not Z, for uniform preparation.
A1 = uniform_preparation(1)
np.testing.assert_allclose(Operator(diffusion(A1)).data, [[0, 1], [1, 0]], atol=1e-12)
print(oracle_for_targets(3, ['001']).draw('text'))
print(diffusion(uniform_preparation(3)).draw('text'))
print(json.dumps({key: report['uniform'][key] for key in
    ('iterations','initial_success','exact_success','sampled_success','state_error_l2')}, indent=2))


## 2. Amplification and overshooting

The five-qubit example marks `00011` and `10101`, so $p=2/32$. The classical curve samples independently with replacement: $1-(1-p)^q$. It uses $q=k+1$ predicate checks to account for a final candidate verification after $k$ quantum marking calls. State preparation, its inverse, diffusion, and gate synthesis are additional quantum costs. This is a query-model comparison, not a simulator runtime or hardware advantage benchmark.


In [ ]:
sweep = report['overshoot_sweep']
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot([r['iterations'] for r in sweep], [r['quantum_success'] for r in sweep], 'o-', label='Exact amplification')
ax.plot([r['iterations'] for r in sweep], [r['classical_success_k_plus_one_checks'] for r in sweep], 's--', label='Independent classical sampling, q=k+1')
u = report['uniform']
ax.errorbar(u['iterations'], u['sampled_success'], yerr=u['hoeffding_half_width'], fmt='x', color='black', label='Aer shots; Hoeffding bound')
ax.set(xlabel='Quantum marking calls k', ylabel='Success probability', ylim=(-.04, 1.04))
ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout(); plt.show()


## 3. General state preparation

A nonuniform three-qubit circuit with RY, H, CX and RZ gates prepares complex amplitudes. Reflecting about $|+\rangle$ would implement the wrong algorithm. The same $A$ and $A^\dagger$ must appear in every diffuser. The independent reference applies $v\mapsto O_\chi v\mapsto2|\psi\rangle\langle\psi|O_\chi v-O_\chi v$ directly in NumPy; it does not call the circuit's oracle or diffuser constructors.


In [ ]:
A = QuantumCircuit(3)
A.ry(.4, 0); A.ry(.9, 1); A.h(2); A.cx(1, 2); A.rz(.6, 0)
print(A.draw('text'))
print(json.dumps(report['nonuniform'], indent=2))


## 4. Reversible 4-Queens constraints and clean workspace

Two bits encode each row's column, with the first row occupying the most significant pair. For every row pair $(i,j)$, a flag records

$$[c_i=c_j]\lor[|c_i-c_j|=j-i].$$

The oracle computes six pair flags, flips phase only when all are zero, and reverses the computation. Pair predicates enumerate at most 16 column pairs, **not complete solutions**. All 256 board assignments are checked in a random complex superposition against a separate classical predicate. Workspace leakage is measured after amplification; the diffuser acts only on the eight system qubits.

Brute-force enumeration and a separate backtracking solver supply reference solutions and known $M=2$. Those classical costs are recorded. Knowing $M$ for the iteration choice does not establish an end-to-end advantage over structured backtracking.


In [ ]:
q4 = report['queens4']
print(json.dumps({'reference': q4['classical'], 'constraint_run': q4['constraint_run']}, indent=2))
for name in ('queens4_lookup_oracle', 'queens4_constraint_oracle', 'queens4_constraint_complete_run'):
    print(name, report['resources'][name])
# These are all-to-all logical u/CX counts, without device mapping or noise.


## 5. Five Queens: retain the encoding distinction

Five rows need three bits each: the binary register contains $2^{15}=32768$ assignments, while only $5^5=3125$ use columns 0–4. Ten assignments are solutions. Uniform initialization therefore has $p=10/32768$, not $10/3125$.

This retained example marks the ten classically enumerated solutions using a lookup phase oracle. Its simulation checks the amplification law and bit encoding; it does not implement a scalable constraint search. The 4-Queens constraint oracle above supplies the separate compute–phase–uncompute demonstration.


In [ ]:
q5 = report['queens5']
print(json.dumps({'classical_reference': q5['classical'],
    'execution': {k:v for k,v in q5['lookup_run'].items() if k != 'counts'},
    'lookup_oracle_resources': report['resources']['queens5_lookup_oracle']}, indent=2))


## 6. Acceptance and scope

Each executed case must agree with the independent complex statevector to $10^{-9}$ in L2 norm, match $P_k$ to $10^{-10}$, and return workspace leakage below $10^{-12}$. Shot checks use the per-run Hoeffding tolerance $\sqrt{\log(2/\delta)/(2S)}$, with $S=4096$ and $\delta=0.001$; this is not a simultaneous interval for every histogram bar. The plotted bound is a sampling tolerance, not hardware uncertainty.

Compiled counts use the declared `u`/CX basis, optimization level 0, and no device routing. They distinguish oracle implementation cost from abstract query count. Ideal simulation establishes neither noise robustness nor wall-clock speedup. The [README](README.md) contains reproduction commands, measured evidence, and bounded extension targets.

Primary references: [Brassard et al., amplitude amplification](https://arxiv.org/abs/quant-ph/0005055) and [IBM's `grover_operator` documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.grover_operator).
